# 03m — Views definitivas para o treino do Random Forest (multiclasse, 7 classes)

Este notebook **não treina nada**. Ele cria duas **views** no ClickHouse, prontas para ler e treinar o RF multiclasse, com as mesmas decisões do notebook 03 (binário) e as particularidades do multiclasse:

| Decisão | O que é feito nas views |
|---|---|
| **Classes** | Só as que existem **no treino e no teste** (comparando o nome sem o prefixo `DrDoS_`, como no notebook 02): `BENIGN` mais os tipos de ataque em comum. As classes só do treino (`DNS`, `NTP`, `SNMP`, `SSDP`, `TFTP`, `WebDDoS`) e a só do teste (`Portmap`) **ficam de fora**, porque não dá para avaliá-las com este protocolo. A lista é calculada na hora e conferida na tela |
| **Resposta `y`** | O **código** da classe: 0 = `BENIGN` e, depois, os tipos de ataque em ordem alfabética (a mesma ordem usada nos notebooks de busca). A tabela `prep_classes_rf_multi` guarda a correspondência código ↔ nome |
| **Valores nulos** | Substituídos pela **mediana do treino**, igual ao `PreProc.transformar(df, "rf")` do notebook 02. As colunas `<feature>_ausente` continuam como atributos |
| **Atributos** | A **seleção multiclasse** do notebook 02 (`tarefa = 'multi'`, decisão `mantida`), menos a ablação abaixo |
| **Ablação (colunas de AUC alto)** | Ficam de fora `bwd_packets_s`, `inbound` e `total_backward_packets`: **as mesmas três do binário** (e os indicadores `_ausente` delas, se existirem). Veja a nota abaixo |
| **Duplicatas treino × teste (opção A)** | O treino fica **completo**. O teste só tem as linhas com `sobrepoe_treino = 0` |

> **Nota sobre os "atalhos suspeitos" do multiclasse.** O notebook 02 marcou **muito mais colunas** como `atalho_suspeito` no multiclasse do que no binário (força de AUC ≥ 0,80 para **uma** classe contra as outras). No multiclasse isso é esperado: um tipo de ataque específico (por exemplo, `flow_duration` para um protocolo) se separa bem do resto sem que a coluna seja um "atalho" do conjunto de dados. Retirar todas deixaria o modelo sem os seus atributos mais informativos e a comparação "com × sem" deixaria de dizer algo sobre atalhos. Por isso a ablação padrão é a **mesma lista do binário**, definida **antes** de ver qualquer resultado do multiclasse. A célula 1.1 mostra a lista completa marcada pelo 02, e a variável `ATALHOS_ABLACAO` pode ser mudada (a versão `03mb` não retira nada).

**Views criadas** (nada existente é alterado; só são criadas views e tabelas de auditoria com prefixo `prep_`):

| Objeto | Conteúdo |
|---|---|
| `prep_vw_rf_multi_treino` | `id_linha`, `y` (código da classe), atributos sem nulos e indicadores `_ausente` |
| `prep_vw_rf_multi_teste_sem_sobrep` | Igual, só com as linhas do teste **sem sobreposição** com o treino e só das classes em comum |
| `prep_views_rf_multi` | Auditoria: cada atributo da seleção multiclasse, se foi usado ou removido, e a mediana usada |
| `prep_classes_rf_multi` | Código, nome e número de linhas de cada classe no treino e no teste |

**Como ler no Python** (a coluna `y` é a resposta; `id_linha` **não** entra na entrada do modelo):
```python
tr = ler_rf_multi("treino", mod=20, resto=0)      # 1/20 do treino, determinístico
X, y = tr.drop(columns=["id_linha", "y"]), tr["y"]
```

**Antes de rodar:** o notebook 02 (v8.10 ou mais novo) precisa ter terminado (tabelas `prep_final_treino`, `prep_final_teste`, `prep_parametros` e `prep_selecao_atributos`). A conexão é a mesma do 02: coloque este arquivo **na mesma pasta** do `.env`. A trava de segurança é a mesma: só é possível escrever em tabelas e views `prep_*`.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas mínimas ----------
import numpy as np
from datetime import datetime

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def blocos_de(lista, n=20):
    return [lista[i:i + n] for i in range(0, len(lista), n)]

## 1. O que entra e o que sai (decisões desta etapa)

A célula abaixo lê do notebook 02: a **seleção multiclasse** de atributos, as **medianas do treino** (`prep_parametros`) e quais atributos têm indicador `_ausente`. Calcula **quais classes existem nos dois dias** e monta a lista final de atributos:

**atributos finais = seleção multiclasse − colunas da ablação**

- A lista de ablação (`ATALHOS_ABLACAO`) é a mesma do binário. **Decida a lista antes de ver resultados** e não a mude depois.
- A célula **mostra** os atalhos que o notebook 02 marcou para o multiclasse (`atalho_suspeito = 1`) e **avisa** que a sua lista é diferente (o que é esperado aqui, pelo motivo explicado na introdução).
- Uma coluna da lista que não esteja na seleção multiclasse é ignorada, com aviso.
- Para cada atributo com ausentes, o indicador `_ausente` **sai junto** com a coluna de origem.
- As classes são as que têm linhas **no treino e no teste sem sobreposição**; o nome é comparado **sem o prefixo `DrDoS_`**.

In [5]:
# ---------- 1.1 Parâmetros e leitura do que o notebook 02 gravou ----------
ATALHOS_ABLACAO = globals().get("ATALHOS_ABLACAO", ["bwd_packets_s", "inbound", "total_backward_packets"])   # as mesmas colunas do binário, definidas ANTES de ver resultados do multiclasse
CLASSES_ATAQUE = globals().get("CLASSES_ATAQUE", None)          # None = calcula: tipos de ataque com linhas no treino e no teste (sem sobreposição)
VIEW_TREINO, VIEW_TESTE, TAB_AUD, TAB_CLASSES = "prep_vw_rf_multi_treino", "prep_vw_rf_multi_teste_sem_sobrep", "prep_views_rf_multi", "prep_classes_rf_multi"

faltam = [t for t in ("prep_final_treino", "prep_final_teste", "prep_parametros", "prep_selecao_atributos") if not existe(t)]
assert not faltam, f"Rode antes o notebook de pré-processamento (02): faltam as tabelas {faltam}"

SEL_MULTI = consultar("SELECT feature FROM prep_selecao_atributos WHERE tarefa = 'multi' AND decisao = 'mantida' ORDER BY posicao")["feature"].tolist()
assert SEL_MULTI, "a seleção multiclasse (tarefa = 'multi') está vazia em prep_selecao_atributos: rode a seção 12 do notebook 02"
par = consultar_num("SELECT coluna, mediana FROM prep_parametros WHERE usa_indicador = 1")
MEDIANA = dict(zip(par.coluna, par.mediana))
cols_final = consultar("SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = 'prep_final_treino'")["name"].tolist()
COM_IND = {c[:-len("_ausente")] for c in cols_final if c.endswith("_ausente")}
assert set(MEDIANA) == COM_IND, f"as colunas com mediana em prep_parametros ({sorted(MEDIANA)}) não coincidem com as colunas *_ausente de prep_final_treino ({sorted(COM_IND)})"
assert not any(pd.isna(v) for v in MEDIANA.values()), "há coluna com indicador mas sem mediana em prep_parametros"

marcadas = consultar("SELECT feature FROM prep_selecao_atributos WHERE tarefa = 'multi' AND decisao = 'mantida' AND atalho_suspeito = 1 ORDER BY posicao")["feature"].tolist()
print("Seleção multiclasse do notebook 02:", len(SEL_MULTI), "atributos")
print("Atalhos que VOCÊ definiu para a ablação (iguais aos do binário):", ATALHOS_ABLACAO)
print(f"Atalhos marcados pelo notebook 02 no multiclasse ({len(marcadas)}):", marcadas)
if set(ATALHOS_ABLACAO) != set(marcadas): print(f"⚠ as duas listas são diferentes ({len(marcadas)} marcadas no 02 contra {len(ATALHOS_ABLACAO)} na ablação): esperado no multiclasse, veja a nota da introdução. A versão 03mb mantém todas.")
fora = [c for c in ATALHOS_ABLACAO if c not in SEL_MULTI]
if fora: print("⚠ não estão na seleção multiclasse e serão ignorados:", fora)
RETIRADOS = [c for c in SEL_MULTI if c in set(ATALHOS_ABLACAO)]
FEATS = [c for c in SEL_MULTI if c not in set(ATALHOS_ABLACAO)]
IND = [c for c in FEATS if c in MEDIANA]
assert FEATS, "nenhum atributo restou depois da ablação"
print(f"\nRetirados pela ablação ({len(RETIRADOS)}): {RETIRADOS}")
print(f"Atributos finais: {len(FEATS)} (+ {len(IND)} indicadores _ausente: {[c + '_ausente' for c in IND]})")
print("Medianas do treino usadas na imputação:", {c: MEDIANA[c] for c in IND})

# ---- classes: as que existem no treino e no teste (nome sem o prefixo DrDoS_) ----
nome_tipo = lambda t: re.sub(r"^DrDoS_", "", str(t))
def contar(tab, onde=""):
    d = consultar_num(f"SELECT label_multi AS tipo, count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "") + " GROUP BY label_multi")
    d["tipo"] = d["tipo"].map(nome_tipo); d = d.groupby("tipo", as_index=False)["n"].sum(); return {r.tipo: int(r.n) for r in d.itertuples()}
N_TREINO, N_TESTE = contar("prep_final_treino"), contar("prep_final_teste", "sobrepoe_treino = 0")        # o teste conta SEM sobreposição (as linhas que valem)
assert "BENIGN" in N_TREINO and "BENIGN" in N_TESTE, "BENIGN precisa existir no treino e no teste"
ATAQUES = sorted(CLASSES_ATAQUE) if CLASSES_ATAQUE else sorted(t for t in N_TESTE if t != "BENIGN" and t in N_TREINO)
assert ATAQUES and all(t in N_TREINO and t in N_TESTE for t in ATAQUES), "toda classe de ataque precisa existir no treino e no teste"
CLASSES = ["BENIGN"] + ATAQUES; COD = {c: i for i, c in enumerate(CLASSES)}
tab_cls = pd.DataFrame({"código": [COD[c] for c in CLASSES], "classe": CLASSES, "linhas no treino": [N_TREINO[c] for c in CLASSES], "linhas no teste (sem sobreposição)": [N_TESTE[c] for c in CLASSES]})
print(f"\n== {len(CLASSES)} classes (BENIGN + {len(ATAQUES)} tipos de ataque presentes nos dois dias) =="); display(tab_cls)
FORA_TREINO, FORA_TESTE = sorted(set(N_TREINO) - set(CLASSES)), sorted(set(N_TESTE) - set(CLASSES))
print("Fora das views (só no treino):", {t: N_TREINO[t] for t in FORA_TREINO}); print("Fora das views (só no teste): ", {t: N_TESTE[t] for t in FORA_TESTE})

Seleção multiclasse do notebook 02: 45 atributos
Atalhos que VOCÊ definiu para a ablação (iguais aos do binário): ['bwd_packets_s', 'inbound', 'total_backward_packets']
Atalhos marcados pelo notebook 02 no multiclasse (17): ['avg_fwd_segment_size', 'total_length_of_fwd_packets', 'fwd_packet_length_max', 'flow_packets_s', 'act_data_pkt_fwd', 'flow_iat_mean', 'max_packet_length', 'flow_bytes_s', 'flow_duration', 'total_fwd_packets', 'flow_iat_std', 'flow_iat_max', 'bwd_packets_s', 'inbound', 'total_backward_packets', 'ack_flag_count', 'idle_std']
⚠ as duas listas são diferentes (17 marcadas no 02 contra 3 na ablação): esperado no multiclasse, veja a nota da introdução. A versão 03mb mantém todas.

Retirados pela ablação (3): ['bwd_packets_s', 'inbound', 'total_backward_packets']
Atributos finais: 42 (+ 2 indicadores _ausente: ['flow_packets_s_ausente', 'flow_bytes_s_ausente'])
Medianas do treino usadas na imputação: {'flow_packets_s': 1.3374280923427226, 'flow_bytes_s': 689.5911165696316

,código,classe,linhas no treino,linhas no teste (sem sobreposição)
0,0,BENIGN,50768,56216
1,1,LDAP,30099,112491
2,2,MSSQL,207003,854242
3,3,NetBIOS,19490,91175
4,4,Syn,155515,4887060
5,5,UDP,1075337,1214746
6,6,UDPLag,89066,557


Fora das views (só no treino): {'DNS': 113259, 'NTP': 1112936, 'SNMP': 115316, 'SSDP': 891221, 'TFTP': 4396919, 'WebDDoS': 414}
Fora das views (só no teste):  {'Portmap': 2702}


## 2. Criação das views

**Classes.** A view troca o rótulo (`label_multi`, com o prefixo `DrDoS_` retirado) pelo **código** da classe e só deixa passar as classes escolhidas. As demais linhas não aparecem.

**Valores nulos.** Nas colunas que têm ausentes, a view troca `NULL` pela **mediana do treino**: `ifNull(coluna, mediana)`. O número da mediana vem de `prep_parametros` e fica **escrito dentro da view**, então a view sempre devolve os mesmos valores. A mediana é do **treino** para os dois conjuntos (o teste não calcula a dele), como no `PreProc`.

**Ablação.** As colunas retiradas simplesmente **não aparecem** na view.

**Opção A.** A view de teste filtra `sobrepoe_treino = 0`. A view de treino não filtra por sobreposição.

**Se o notebook 02 for refeito** e as medianas ou a seleção mudarem, **rode este notebook de novo** para recriar as views.

In [6]:
# ---------- 2.1 Views definitivas e tabelas de auditoria ----------
versao_views = datetime.now().strftime("%Y%m%d_%H%M%S")
def _expr(c):
    return f"ifNull(toFloat64({q(c)}), {float(MEDIANA[c])!r}) AS {q(c)}" if c in MEDIANA else f"toFloat64({q(c)}) AS {q(c)}"
sel_cols = ", ".join([_expr(c) for c in FEATS] + [f"{q(c + '_ausente')} AS {q(c + '_ausente')}" for c in IND])
# `label_multi` é LowCardinality(String) no servidor: o CAST para String (e, no fim, para UInt8) evita que `y` herde LowCardinality, tipo que o ClickHouse proíbe criar numa coluna de view
rot_sql = "replaceRegexpOne(CAST(label_multi AS String), '^DrDoS_', '')"
nomes_sql, cods_sql = ", ".join(lit(c) for c in CLASSES), ", ".join(str(COD[c]) for c in CLASSES)
y_expr, filtro_cls = f"CAST(transform({rot_sql}, [{nomes_sql}], [{cods_sql}], 255) AS UInt8)", f"{rot_sql} IN ({nomes_sql})"

executar(f"DROP VIEW IF EXISTS `{VIEW_TREINO}`")
executar(f"CREATE VIEW {VIEW_TREINO} AS SELECT id_linha, {y_expr} AS y, {sel_cols} FROM prep_final_treino WHERE {filtro_cls}")
executar(f"DROP VIEW IF EXISTS `{VIEW_TESTE}`")
executar(f"CREATE VIEW {VIEW_TESTE} AS SELECT id_linha, {y_expr} AS y, {sel_cols} FROM prep_final_teste WHERE sobrepoe_treino = 0 AND {filtro_cls}")

executar(f"DROP TABLE IF EXISTS `{TAB_AUD}`")
executar(f"CREATE TABLE {TAB_AUD} (posicao UInt16, feature String, situacao String, motivo String, mediana_imputacao Nullable(Float64), tem_indicador UInt8, versao String) ENGINE = MergeTree ORDER BY posicao")
linhas = []
for pos, c in enumerate(SEL_MULTI, 1):
    usada = c not in set(ATALHOS_ABLACAO)
    med = f"{float(MEDIANA[c])!r}" if (usada and c in MEDIANA) else "NULL"
    linhas.append(f"({pos}, {lit(c)}, {lit('usada' if usada else 'removida')}, {lit('' if usada else 'ablação: AUC alto (atalho suspeito)')}, {med}, {int(usada and c in MEDIANA)}, {lit(versao_views)})")
executar(f"INSERT INTO {TAB_AUD} VALUES " + ", ".join(linhas))

executar(f"DROP TABLE IF EXISTS `{TAB_CLASSES}`")
executar(f"CREATE TABLE {TAB_CLASSES} (codigo UInt8, classe String, linhas_treino UInt64, linhas_teste_sem_sobrep UInt64, versao String) ENGINE = MergeTree ORDER BY codigo")
executar(f"INSERT INTO {TAB_CLASSES} VALUES " + ", ".join(f"({COD[c]}, {lit(c)}, {N_TREINO[c]}, {N_TESTE[c]}, {lit(versao_views)})" for c in CLASSES))
print(f"✔ Views criadas: {VIEW_TREINO} e {VIEW_TESTE}\n✔ Auditoria: {TAB_AUD} ({len(linhas)} atributos) e {TAB_CLASSES} ({len(CLASSES)} classes)  | versão {versao_views}")
print("\nDefinição gravada da view de teste:\n" + str(consultar(f"SHOW CREATE VIEW {VIEW_TESTE}").iloc[0, 0])[:900].replace("\\n", "\n") + " ...")

✔ Views criadas: prep_vw_rf_multi_treino e prep_vw_rf_multi_teste_sem_sobrep
✔ Auditoria: prep_views_rf_multi (45 atributos) e prep_classes_rf_multi (7 classes)  | versão 20261026_010742

Definição gravada da view de teste:
CREATE VIEW railway.prep_vw_rf_multi_teste_sem_sobrep
(
    `id_linha` UInt64,
    `y` UInt8,
    `avg_fwd_segment_size` Float64,
    `total_length_of_fwd_packets` Float64,
    `fwd_packet_length_max` Float64,
    `flow_packets_s` Float64,
    `act_data_pkt_fwd` Float64,
    `flow_iat_mean` Float64,
    `max_packet_length` Float64,
    `flow_bytes_s` Float64,
    `flow_duration` Float64,
    `total_fwd_packets` Float64,
    `flow_iat_std` Float64,
    `flow_iat_max` Float64,
    `bwd_header_length` Float64,
    `bwd_iat_total` Float64,
    `bwd_iat_max` Float64,
    `bwd_iat_mean` Float64,
    `ack_flag_count` Float64,
    `min_seg_size_forward` Float64,
    `bwd_iat_min` Float64,
    `init_win_bytes_forward` Float64,
    `fwd_packet_length_std` Float64,
    `flow_i

## 3. Validação

Confere, de forma independente, que as views fazem o que foi decidido:

1. as colunas são exatamente as esperadas, e **nenhuma** coluna retirada na ablação (nem rótulo, `origem_tabela` ou marca de controle) aparece;
2. o treino tem **todas** as linhas das classes escolhidas em `prep_final_treino`, e o teste tem as linhas dessas classes **sem** sobreposição;
3. `y` só tem os códigos das classes escolhidas, **todas** presentes nos dois conjuntos, com a contagem esperada de cada uma;
4. **não há nulo nem infinito** em nenhum atributo;
5. **equivalência com o `PreProc`:** numa amostra, o resultado da view é **igual** ao que se obtém lendo as linhas originais e aplicando a imputação do RF (completar com a mediana do treino e criar os indicadores), e `y` é o código da classe de cada linha.

In [7]:
# ---------- 3.1 Validação ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

esperadas = ["id_linha", "y"] + FEATS + [c + "_ausente" for c in IND]
proibidas = {"label_original", "label_multi", "label_binary", "origem_tabela", "n_repeticoes", "conflito_rotulo", "conflito_binario", "sobrepoe_treino", "sobrepoe_treino_mesmo_rotulo", "b"} | set(RETIRADOS) | {c + "_ausente" for c in RETIRADOS}
for v in (VIEW_TREINO, VIEW_TESTE):
    cols_v = consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(v)} ORDER BY position")["name"].tolist()
    verifica(f"{v}: colunas = id_linha, y, atributos finais e seus indicadores", cols_v == esperadas, f"({len(cols_v)} colunas)")
    verifica(f"{v}: sem rótulo, marcas de controle nem colunas retiradas na ablação", not (proibidas & set(cols_v)))
    tipo_y = consultar(f"SELECT type FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(v)} AND name = 'y'")["type"].iloc[0]
    verifica(f"{v}: y é UInt8 simples (sem LowCardinality)", tipo_y == "UInt8", f"(tipo = {tipo_y})")

n_tr_esp, n_te_esp = sum(N_TREINO[c] for c in CLASSES), sum(N_TESTE[c] for c in CLASSES)
verifica("Treino: todas as linhas das classes escolhidas em prep_final_treino", n_linhas(VIEW_TREINO) == n_tr_esp, f"({n_tr_esp:,})")
verifica("Teste: só as linhas das classes escolhidas e sem sobreposição com o treino", n_linhas(VIEW_TESTE) == n_te_esp, f"({n_te_esp:,}; o teste completo tem {n_linhas('prep_final_teste'):,})")
for v, esp in ((VIEW_TREINO, N_TREINO), (VIEW_TESTE, N_TESTE)):
    d = consultar_num(f"SELECT y, count() AS n FROM {v} GROUP BY y"); obt = {int(r.y): int(r.n) for r in d.itertuples()}
    verifica(f"{v}: y só com os códigos das classes escolhidas, cada uma com a contagem esperada", obt == {COD[c]: esp[c] for c in CLASSES}, f"({len(obt)} classes)")

for v in (VIEW_TREINO, VIEW_TESTE):
    ruins = {}
    for b in blocos_de(FEATS, 20):
        r = consultar_num(f"SELECT " + ", ".join(f"countIf(isNull({q(c)}) OR isInfinite({q(c)})) AS {q(c)}" for c in b) + f" FROM {v}").iloc[0]
        ruins.update({c: int(n) for c, n in r.items() if n > 0})
    verifica(f"{v}: nenhum nulo nem infinito nos atributos", not ruins, f"({ruins})" if ruins else "")

def _equivale(view, fonte, extra, mod=1500):
    # Reaplica, em pandas, a regra do PreProc.transformar(..., 'rf') às linhas originais e compara com a view (valores e código da classe).
    # A amostra tem linhas sorteadas (1/mod) MAIS linhas com valor ausente (1/100 delas), para a imputação ser de fato exercitada.
    ausente = " OR ".join(f"isNull({q(c)})" for c in IND)
    cond = f"(cityHash64(id_linha) % {mod} = 0" + (f" OR (({ausente}) AND cityHash64(id_linha) % 100 = 1)" if IND else "") + f"){extra} AND {filtro_cls}"
    cols = FEATS + [c + "_ausente" for c in IND]
    v = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in cols) + f" FROM {view} WHERE id_linha IN (SELECT id_linha FROM {fonte} WHERE {cond}) ORDER BY id")
    r = consultar(f"SELECT toString(id_linha) AS id, label_multi AS rotulo, " + ", ".join(q(c) for c in FEATS) + f" FROM {fonte} WHERE {cond} ORDER BY id")
    if len(v) != len(r) or (v["id"].values != r["id"].values).any(): return len(v), 0, False
    X = r[FEATS].apply(pd.to_numeric, errors="coerce").astype("float64").replace([np.inf, -np.inf], np.nan)
    ind = {c + "_ausente": X[c].isna().astype("uint8") for c in IND}
    n_aus = int(pd.DataFrame(ind, index=X.index).max(axis=1).sum()) if IND else 0
    for c in IND: X[c] = X[c].fillna(MEDIANA[c])
    esp = pd.concat([X, pd.DataFrame(ind, index=X.index)], axis=1)
    obt = v[list(esp.columns)].apply(pd.to_numeric, errors="coerce")
    y_esp = r["rotulo"].map(nome_tipo).map(COD).astype(int).values
    return len(v), n_aus, bool(np.allclose(obt.values.astype(float), esp.values.astype(float), rtol=1e-12, atol=1e-12) and (v["y"].astype(int).values == y_esp).all())
for v, fonte, extra, nome in ((VIEW_TREINO, "prep_final_treino", "", "treino"), (VIEW_TESTE, "prep_final_teste", " AND sobrepoe_treino = 0", "teste")):
    n_am, n_aus, igual = _equivale(v, fonte, extra)
    verifica(f"Equivalência com o PreProc (RF) em amostra do {nome}: valores e classe", igual and n_am > 0, f"({n_am:,} linhas comparadas, {n_aus:,} delas com valor ausente imputado)")
    if IND and n_aus == 0: print(f"   ⚠ a amostra do {nome} não teve nenhuma linha com valor ausente: a imputação não foi exercitada nesta conferência.")
verifica("Auditoria: uma linha por atributo da seleção multiclasse", n_linhas(TAB_AUD) == len(SEL_MULTI), f"({n_linhas(TAB_AUD)})")
verifica("Classes: uma linha por classe, com as contagens do treino e do teste", n_linhas(TAB_CLASSES) == len(CLASSES), f"({n_linhas(TAB_CLASSES)})")
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")

✔ prep_vw_rf_multi_treino: colunas = id_linha, y, atributos finais e seus indicadores (46 colunas)
✔ prep_vw_rf_multi_treino: sem rótulo, marcas de controle nem colunas retiradas na ablação 
✔ prep_vw_rf_multi_treino: y é UInt8 simples (sem LowCardinality) (tipo = UInt8)
✔ prep_vw_rf_multi_teste_sem_sobrep: colunas = id_linha, y, atributos finais e seus indicadores (46 colunas)
✔ prep_vw_rf_multi_teste_sem_sobrep: sem rótulo, marcas de controle nem colunas retiradas na ablação 
✔ prep_vw_rf_multi_teste_sem_sobrep: y é UInt8 simples (sem LowCardinality) (tipo = UInt8)
✔ Treino: todas as linhas das classes escolhidas em prep_final_treino (1,627,278)
✔ Teste: só as linhas das classes escolhidas e sem sobreposição com o treino (7,216,487; o teste completo tem 20,364,525)
✔ prep_vw_rf_multi_treino: y só com os códigos das classes escolhidas, cada uma com a contagem esperada (7 classes)
✔ prep_vw_rf_multi_teste_sem_sobrep: y só com os códigos das classes escolhidas, cada uma com a contagem e

## 4. Resumo e leitura em lotes

Mostra o tamanho de cada classe no treino e no teste e **quantas linhas de cada classe a opção A tirou do teste**. Se sobrarem poucas linhas de uma classe no teste sem sobreposição, as métricas dela ficam menos estáveis, e isso precisa aparecer no TCC.

A função `ler_rf_multi` lê as views em lotes (`mod` e `resto`), porque o treino tem milhões de linhas e não deve ser carregado de uma vez. Cada lote é a fração `resto/mod` das linhas, sorteada de forma **determinística** por `id_linha`.

In [8]:
# ---------- 4.1 Resumo e leitura em lotes ----------
d = consultar_num(f"SELECT 'treino' AS conjunto, y, count() AS linhas FROM {VIEW_TREINO} GROUP BY y "
                  f"UNION ALL SELECT 'teste (sem sobreposição)', y, count() FROM {VIEW_TESTE} GROUP BY y")
t = d.pivot_table(index="y", columns="conjunto", values="linhas", aggfunc="sum", fill_value=0).astype(int)
comp = consultar_num(f"SELECT {y_expr} AS y, count() AS linhas FROM prep_final_teste WHERE {filtro_cls} GROUP BY y"); t["teste (completo)"] = comp.set_index(comp.y.astype(int)).linhas.astype(int)
t = t[["treino", "teste (completo)", "teste (sem sobreposição)"]]
t["removidas pela opção A"] = t["teste (completo)"] - t["teste (sem sobreposição)"]; t["% removido"] = (100 * t["removidas pela opção A"] / t["teste (completo)"]).round(1)
t["% do teste"] = (100 * t["teste (sem sobreposição)"] / t["teste (sem sobreposição)"].sum()).round(3)
t.index = [CLASSES[int(i)] for i in t.index]
print("== Linhas por classe =="); display(t)
menor = t["teste (sem sobreposição)"].idxmin()
print(f"\nMenor classe no teste sem sobreposição: {menor} ({int(t.loc[menor, 'teste (sem sobreposição)']):,} linhas)" + ("  ⚠ poucas linhas: as métricas dessa classe ficam instáveis; relate isso." if t.loc[menor, "teste (sem sobreposição)"] < 1000 else ""))
print(f"Atributos: {len(FEATS)} + {len(IND)} indicadores = {len(FEATS) + len(IND)} colunas de entrada. RAM estimada p/ o treino inteiro das classes escolhidas em float64: {n_tr_esp * (len(FEATS) + len(IND)) * 8 / 2**30:.2f} GiB (por isso, ler em lotes).")

def ler_rf_multi(conjunto, mod=None, resto=0, limite=None):
    # conjunto: 'treino' ou 'teste'. Lê a fração resto/mod das linhas (determinística); sem mod, lê tudo (cuidado com o treino e o teste).
    view = VIEW_TREINO if conjunto == "treino" else VIEW_TESTE
    onde = f" WHERE cityHash64(id_linha) % {int(mod)} = {int(resto)}" if mod else ""
    return consultar_num(f"SELECT * FROM {view}{onde}" + (f" LIMIT {int(limite)}" if limite else ""))

for conj in ("treino", "teste"):                                   # teste rápido de leitura (1/500 de cada conjunto)
    a = ler_rf_multi(conj, mod=500, resto=0)
    assert not a.drop(columns=["id_linha"]).isna().any().any(), "NaN na leitura"
    print(f"Leitura de teste ({conj}, 1/500): {a.shape[0]:,} linhas x {a.shape[1]} colunas, sem NaN | classes: { {CLASSES[int(k)]: int(v) for k, v in a['y'].value_counts().sort_index().items()} }")

== Linhas por classe ==


conjunto,treino,teste (completo),teste (sem sobreposição),removidas pela opção A,% removido,% do teste
BENIGN,50768,56965,56216,749,1.3,0.779
LDAP,30099,1915122,112491,1802631,94.1,1.559
MSSQL,207003,5787453,854242,4933211,85.2,11.837
NetBIOS,19490,3657497,91175,3566322,97.5,1.263
Syn,155515,4891500,4887060,4440,0.1,67.721
UDP,1075337,3867155,1214746,2652409,68.6,16.833
UDPLag,89066,1873,557,1316,70.3,0.008



Menor classe no teste sem sobreposição: UDPLag (557 linhas)  ⚠ poucas linhas: as métricas dessa classe ficam instáveis; relate isso.
Atributos: 42 + 2 indicadores = 44 colunas de entrada. RAM estimada p/ o treino inteiro das classes escolhidas em float64: 0.53 GiB (por isso, ler em lotes).
  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "id_linha",\n\t\t\t"type": "UInt64"\n\t\t},\n\t\t{\n\t\t\t"name": "y",\n\t\t\t"); nova tentativa em 5s...
Leitura de teste (treino, 1/500): 3,321 linhas x 46 colunas, sem NaN | classes: {'BENIGN': 110, 'LDAP': 64, 'MSSQL': 418, 'NetBIOS': 37, 'Syn': 280, 'UDP': 2236, 'UDPLag': 176}
Leitura de teste (teste, 1/500): 14,262 linhas x 46 colunas, sem NaN | classes: {'BENIGN': 109, 'LDAP': 236, 'MSSQL': 1627, 'NetBIOS': 178, 'Syn': 9689, 'UDP': 2423}
